# Measure mesh error before refining

We solve a diffusion problem on a unit square using ordinary Geometry, Model,
Mesh, Plan and Result objects. Every modeling and analysis step is in this notebook.
A small linear-solver residual is not a spatial-accuracy certificate.

Use a Python kernel containing the **development checkout** of Eqiora and NumPy.
The released 0.1.3 wheel does not contain this API. For example, install the checkout
into the kernel environment with `python -m pip install /path/to/eqiora`.
Hosted Colab execution has not been verified.


In [ ]:
from pathlib import Path
import numpy as np
import eqiora as q


## Define the physical problem

Let X=x/(1 m), Y=y/(1 m). The exact solution is
u=(1 K) X(1-X)Y(1-Y), so u=0 on the square boundary and
-div(grad(u))=(2 K/m²)[X(1-X)+Y(1-Y)].
The exact squared H1 seminorm is 1/45 K², by integrating the two squared derivatives.
We use this known solution to validate the study; an unknown engineering solution
would not supply this exact reference.


In [ ]:
graph = q.geometry.GeometryGraph()
rectangle = graph.rectangle(x_bounds=(0.0, 1.0), y_bounds=(0.0, 1.0))
geometry = graph.build(rectangle, named_topology={
    "body": rectangle.region,
    "outer": rectangle.boundaries,
})


In [ ]:
source = """
public component Diffusion(
    support body:volume(ambient_dimension=2),
    support outer:boundary(parent=body)
) {
    coordinate x:m on body from body[0];
    coordinate y:m on body from body[1];
    variable X:1 on body;
    variable Y:1 on body;
    variable u:K on body in h1;
    relation x_value on body { X=coordinate(0)/1[m]; }
    relation y_value on body { Y=coordinate(1)/1[m]; }
    relation balance on body { -div(grad(u))=2[K/m^2]*(X*(1-X)+Y*(1-Y)); }
    relation fixed on outer { trace(u)=0[K]; }
    observable energy:K^2=integral(partial(u,wrt=x)^2+partial(u,wrt=y)^2,measure(body));
}
"""
model = q.compile(source=source, entry="Diffusion", geometry=geometry, bindings={
    "body": geometry.selection("body"),
    "outer": (geometry.selection("outer"), geometry.selection("body")),
})
field = model.field("definition.u")
energy = model.observable("definition.energy")


## Hold the solver policy fixed

Only the number of mesh cells changes. P1 represents a continuous, piecewise-affine
field. The source is integrated through the existing Plan assembly; no separate
matrix assembler or solver is introduced here.


In [ ]:
solver = q.solve.Linear(
    algorithm=q.solve.LinearSolver.BiConjugateGradientStabilized,
    preconditioner=q.solve.Preconditioner.Identity,
    reduction=q.solve.Reduction.Reproducible,
    provider=q.solve.SolverProvider.reference(),
    relative_tolerance=1e-12,
    absolute_tolerance=1e-14,
    maximum_iterations=1000,
)


## Solve, then query the residual estimator

The indicator contains the element source residual and the normal-gradient jump
across interior edges. The global estimate is the Euclidean norm of the cell
indicators. Its physical unit is K in this two-dimensional problem.

The query supports constant positive scalar diffusion, homogeneous essential
boundaries and triangular P1 elements. It is an H1-seminorm **estimate**, not a
certified upper bound. The explicit three-point-per-axis Duffy rule integrates
this polynomial source squared exactly. General sources require their own
quadrature-accuracy study.


In [ ]:
history = []
for n in (2, 4, 8):
    mesh = q.meshing.generate(q.meshing.resolve(
        geometry, q.meshing.AffineTriangleMesher(cells=(n, n))))
    plan = q.resolve(model, mesh=mesh, spatial=(q.fem.P1().at(model.domain("body")),), solve=solver)
    result = q.run(plan)
    indicators, dimension = result.discretization_error_indicators(field, quadrature_points=3)
    indicators = np.asarray(indicators)
    assert dimension == q.Dimension(temperature=1)
    assert not indicators.flags.writeable
    estimate = float(np.linalg.norm(indicators))
    discrete_energy = result.observe(energy, quadrature_points=2).value
    # Exact Galerkin orthogonality: |u-u_h|_H1² = |u|_H1² - |u_h|_H1².
    error_squared = 1/45 - discrete_energy
    assert error_squared > 0
    history.append(dict(n=n, mesh=mesh, plan=plan, result=result,
                        indicators=indicators, dimension=dimension,
                        estimate=estimate, error_squared=error_squared))


## Check against independent discrete values

On this uniformly triangulated square the exact P1 matrix is the five-point
stencil (4,-1,-1,-1,-1). Integrating the six adjacent vertex hats gives the load
h²[2X(1-X)+2Y(1-Y)-2h²/3]. Exact rational elimination gives the three squared
errors below. These expectations are derived from the stencil, not from the
estimator output. The native energy Observable and the estimator use separate
queries on the same accepted Result.


In [ ]:
expected_errors_squared = (131/11520, 20377/5898240, 15567421/17112760320)
for row, expected in zip(history, expected_errors_squared, strict=True):
    assert abs(row["error_squared"] - expected) < 1e-12
    print(f'n={row["n"]:2d}  cells={row["mesh"].cell_count:3d}  '
          f'H1 error={np.sqrt(row["error_squared"]):.6f} K  '
          f'estimate={row["estimate"]:.6f} K')
assert history[2]["estimate"] < history[1]["estimate"] < history[0]["estimate"]


## Retain every mesh and accepted result

A Plan retains its exact Mesh and numerical settings. Save each Plan and Result,
then reopen the final pair and repeat the same explicitly requested estimate.
This keeps mesh revisions separate and prevents reuse of an old field on a new mesh.
The example's `history` also retains each estimate, its Dimension and its Result.


In [ ]:
output = Path("mesh-error-study")
output.mkdir(exist_ok=True)
for row in history:
    stem = output / f'cells-{row["n"]}'
    row["plan"].write(str(stem) + ".eqplan")
    row["result"].write(str(stem) + ".eqresult")
last = history[-1]
reopened_plan = q.Plan.read(output / "cells-8.eqplan")
reopened = q.Result.read(reopened_plan, output / "cells-8.eqresult")
repeated, repeated_dimension = reopened.discretization_error_indicators(field, quadrature_points=3)
np.testing.assert_array_equal(np.asarray(repeated), last["indicators"])
assert repeated_dimension == last["dimension"]


## What remains

This is a uniform-refinement accuracy study. It does not mark local cells,
refine them adaptively, transfer a field, or certify a tolerance. Those operations
must preserve the same Geometry/Mesh/Field ownership before an adaptive loop can
claim more. Do not mistake the estimate, the exact manufactured-solution error,
or the algebraic residual for each other.
